# Pipeline 4: Citizenship Integration

This pipeline incorporates the verified `NAC_NATIONALITY` from the NMAT dataset and the profiling data from `pseudo_citizenship_profiling_FINAL.csv` into a unified set of columns in `NMAT_Ultima.parquet`.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path("dataset")
ULTIMA_PATH = ROOT / "NMAT_Ultima.parquet"
PSEUDO_PATH = ROOT / "pseudo_citizenship_profiling_FINAL.csv"

print(f"Loading {ULTIMA_PATH}...")
df = pd.read_parquet(ULTIMA_PATH)

In [ ]:
print(f"Loading {PSEUDO_PATH}...")
pc_df = pd.read_csv(PSEUDO_PATH, usecols=["APPNO_CLEAN", "override_applied", "pseudo_citizenship", "name_based_assessment"], dtype=str)
pc_df["APPNO_CLEAN"] = pc_df["APPNO_CLEAN"].str.strip()
pc_df = pc_df.drop_duplicates(subset=["APPNO_CLEAN"])

In [ ]:
print("Merging pseudo-citizenship data...")
df["APPNO_CLEAN_str"] = df["APPNO_CLEAN"].astype(str).str.strip()
df = df.merge(pc_df, left_on="APPNO_CLEAN_str", right_on="APPNO_CLEAN", how="left", suffixes=("", "_pseudo"))
df = df.drop(columns=["APPNO_CLEAN_str", "APPNO_CLEAN_pseudo"], errors="ignore")

In [ ]:
def determine_citizenship(row):
    nat = str(row.get('NAC_NATIONALITY', '')).strip().title()
    # If it's a known Filipino variant or empty, we check the pseudo lookup
    filipino_variants = ['', 'Nan', 'None', 'Filipino', 'Philippines', 'Fil-Am', 'Fil', 'Ph', 'Filipina']
    
    if nat and nat not in filipino_variants:
        return nat, 'Verified Foreigner'
    
    pseudo = str(row.get('pseudo_citizenship', '')).strip().title()
    override = str(row.get('override_applied', '')).strip().upper()
    
    if pseudo and pseudo not in filipino_variants and override == 'FOREIGN':
        return pseudo, 'Likely Foreigner'
    
    return 'Filipino', 'Filipino'

print("Applying citizenship logic...")
result = df.apply(determine_citizenship, axis=1)
df["CITIZENSHIP_FINAL"] = [r[0] for r in result]
df["FOREIGNER_STATUS"] = [r[1] for r in result]

In [ ]:
print("Citizenship Distribution:")
print(df["CITIZENSHIP_FINAL"].value_counts().head(20))
print("\nForeigner Status Distribution:")
print(df["FOREIGNER_STATUS"].value_counts())

In [ ]:
print(f"Saving updated parquet back to {ULTIMA_PATH}...")
df.to_parquet(ULTIMA_PATH, index=False)
print("Done!")